# 策略研发全流程 Pipeline

本 Notebook 是量化策略研发的统一核心入口，严格遵循 `strategy_rules.md` 中的样本内外划分与无前视偏差规则：
- **IS_Train (滚动训练集)**: `2021-01-01` ~ `2023-09-30`，仅用于 Rolling WFV 内部早停与调参。
- **IS_Test (核心测试集)**: `2023-10-01` ~ `2024-09-01`，使用最近 4-Fold 集成进行验证，决定策略是否可以进入实盘预备。
- **OOS (严格样本外)**: `2024-09-01` 之后，作为盲区进行最后一道防线检验。

In [1]:
import sys
import logging
from pathlib import Path

# 配置项目根目录
ROOT = Path("/root/autodl-tmp/Strategy")
sys.path.insert(0, str(ROOT.parent))

logging.basicConfig(level=logging.INFO, force=True)
for name in ("Strategy", "Strategy.model", "Strategy.backtest"):
    logging.getLogger(name).setLevel(logging.INFO)

from Strategy import config


## 1. 数据准备 (Label & Factors)
在全市场范围内生成所需的 Label 和因子。本 Notebook **训练默认标签 `OPEN930_1000`**：买入价为连续竞价 **首根 K 线**（本仓库分钟数据 `time=931`，与 `OPEN0935` 同理均为真实 bar）→ T+1 **10:00** 卖出（见 `label_generator`）。依赖 **`min_data`** 与 **`Daily_data`**。因子对齐 **T-1 收盘后** 可得。可改用 **`OPEN0935_1000`** 或 **`CLOSE_PRECLOSE`**。


In [2]:
from Strategy.label.label_generator import generate_and_save_open930_1000_label

LABEL_TAG = "OPEN930_1000"
# 需 min_data 分钟线；Daily_data: CLOSE/PRE_CLOSE（除权）+ LIMIT_UP + LIMIT_DOWN（默认涨跌停剔除）
# 可选: start_date=20210104, end_date=20251231；调试可无涨跌停: use_limit_price_tables=False
open930_label_path = generate_and_save_open930_1000_label()
print(f"[{LABEL_TAG}] Label 已生成: {open930_label_path}")

# 其他示例:
# from Strategy.label.label_generator import generate_and_save_open0935_1000_label, generate_and_save_close_preclose_label
# generate_and_save_open0935_1000_label(save_price_tables=True)
# generate_and_save_close_preclose_label()


INFO:Strategy.label.label_generator:开始计算分钟 open 宽表 time=931, 共 1261 个交易日


Open@931:   0%|          | 0/1261 [00:00<?, ?day/s]

INFO:Strategy.label.label_generator:open 宽表计算完成 time=931, shape=(1261, 5380)
INFO:Strategy.label.label_generator:开始计算分钟 open 宽表 time=1000, 共 1261 个交易日


Open@1000:   0%|          | 0/1261 [00:00<?, ?day/s]

INFO:Strategy.label.label_generator:open 宽表计算完成 time=1000, shape=(1261, 5380)
INFO:Strategy.label.label_generator:OPEN930_1000 Label 除权调整已应用 (CLOSE / PRE_CLOSE), 共 5380 支股票列对齐
INFO:Strategy.label.label_generator:OPEN930_1000 涨跌停剔除: 买入价涨停=17911 | T+1卖出价跌停=14887
INFO:Strategy.label.label_generator:OPEN930_1000 Label 已保存: /root/autodl-tmp/Strategy/outputs/labels/LABEL_OPEN930_1000.fea, shape=(1261, 5380)
INFO:Strategy.label.label_generator:OPEN930_1000 买入价格 mask 已保存: /root/autodl-tmp/Strategy/outputs/labels/OPEN930_1000.fea


[OPEN930_1000] Label 已生成: /root/autodl-tmp/Strategy/outputs/labels/LABEL_OPEN930_1000.fea


In [3]:
# from Strategy.factor.daily_factor_library import DailyFactorLibraryAdapter

# # 每日频因子需覆盖到当前可用的最新日期
# adapter = DailyFactorLibraryAdapter()
# saved_daily = adapter.compute_and_save_all(
#     start_date=config.IS_TRAIN_START,
#     # end_date=config.IS_TRAIN_END,  # 仅测试时取消注释以加速
# )
# print(f"已保存 {len(saved_daily)} 个日频因子")


In [4]:
# from Strategy.factor.factor_base import FactorRegistry
# import Strategy.factor.minute_derived_factors   # 触发注册
# import Strategy.factor.custom_factors           # 触发注册

# # 计算并保存所有注册的高频特征
# FactorRegistry.compute_all()


## 2. 面板构建与样本拆分
严格按照配置，物理隔离出 `is_train`, `is_test`, `oos`。模型训练时将绝不碰触 `is_test` 和 `oos`。

In [5]:
from Strategy.factor.factor_base import load_all_factors
from Strategy.label.label_generator import load_label
from Strategy.model.trainer import build_panel, split_panel
import pandas as pd

LABEL_TAG = "OPEN930_1000"
factor_dict = load_all_factors()
label_df = load_label(LABEL_TAG)

# 展平构建 Panel
panel = build_panel(factor_dict, label_df)

# 严格切割样本 (依据 config 中定好的常数)
is_train, is_test, oos = split_panel(panel)

print(f"IS_Train: {is_train.shape[0]} rows (截止 {is_train['TRADE_DATE'].max().date()})")
print(f"IS_Test:  {is_test.shape[0]} rows (截止 {is_test['TRADE_DATE'].max().date()})")
print(f"OOS:      {oos.shape[0]} rows (起始 {oos['TRADE_DATE'].min().date()})")


INFO:Strategy.model.trainer:Panel 对齐: 1251 交易日 × 5324 只股票, 134 个因子
INFO:Strategy.model.trainer:正在 concat 135 个 Series ...
INFO:Strategy.model.trainer:Panel 构建完成: shape=(6660324, 137)
INFO:Strategy.model.trainer:panel 日期: [2021-01-18, 2026-03-20]
INFO:Strategy.model.trainer:IS Train: 3497868 rows | IS Test: 1181928 rows | OOS: 1980528 rows


IS_Train: 3497868 rows (截止 2023-09-28)
IS_Test:  1181928 rows (截止 2024-08-30)
OOS:      1980528 rows (起始 2024-09-02)


## 3. IS Train 滚动训练 (Rolling CV)
利用 `is_train` 内部通过时间窗滚动的方式进行训练和早停，并保存模型参数。
与 `strategy_rules.md` 一致：**截面模型**、**`RollingTrainer` 时间块 CV**、神经网络 **`batch_size=1`**（每个优化步仅一个交易日截面；见 `config.NN_TRAINER_BATCH_SIZE`）。


In [6]:
from Strategy.model.rolling_trainer import RollingTrainer
from Strategy.model.trainer import XGBTrainer

rt_xgb = RollingTrainer(
    model_class=XGBTrainer,
    model_kwargs={
        "use_wpcc": True,
    }
)

# 仅在 IS_Train 上执行所有 Fold 的滚动训练
rt_xgb.train_all_folds(is_train)

print("\n========== XGB CV Fold IC ==========")
print(rt_xgb.fold_ic_report())

rt_xgb.save_model(config.SCORE_OUTPUT_DIR / f"rolling_xgb_{LABEL_TAG}.pkl")


INFO:Strategy.model.rolling_trainer:generate_folds: IS Train=[2021-01-01, 2023-09-30], val_months=3 → 11 Fold
INFO:Strategy.model.rolling_trainer:Rolling Val CV: 11 Fold, 特征数=134, IS Train=2021-01-01~2023-09-30
INFO:Strategy.model.rolling_trainer:━━ Fold 1/11: Val=[2021-01-01, 2021-03-31]  Train=IS Train-Val ━━
INFO:Strategy.model.rolling_trainer:  Train: 3242316 行 (609 日) | Val: 255552 行 (48 日)
INFO:Strategy.model.trainer:特征数=134
INFO:Strategy.model.trainer:训练样本 n=2855299; label std=0.0350967; 特征 NaN=0.15%
INFO:Strategy.model.trainer:使用 WPCC 自定义目标函数


[0]	train-rmse:0.50043	train-wpcc:-0.03575	val-rmse:0.50018	val-wpcc:-0.05686
[83]	train-rmse:0.50043	train-wpcc:-0.13259	val-rmse:0.50019	val-wpcc:-0.04206


INFO:Strategy.model.trainer:训练完成. best_iteration=3
INFO:Strategy.model.rolling_trainer:  Fold 1 Val  IC=0.0421  ICIR=0.8626  RankIC=0.0575  days=48
INFO:Strategy.model.rolling_trainer:━━ Fold 2/11: Val=[2021-04-01, 2021-06-30]  Train=IS Train-Val ━━
INFO:Strategy.model.rolling_trainer:  Train: 3178428 行 (597 日) | Val: 319440 行 (60 日)
INFO:Strategy.model.trainer:特征数=134
INFO:Strategy.model.trainer:训练样本 n=2799039; label std=0.0353843; 特征 NaN=0.14%
INFO:Strategy.model.trainer:使用 WPCC 自定义目标函数


[0]	train-rmse:0.50057	train-wpcc:-0.03935	val-rmse:0.49878	val-wpcc:-0.03860
[94]	train-rmse:0.50057	train-wpcc:-0.14120	val-rmse:0.49878	val-wpcc:-0.05232


INFO:Strategy.model.trainer:训练完成. best_iteration=14
INFO:Strategy.model.rolling_trainer:  Fold 2 Val  IC=0.0523  ICIR=0.9736  RankIC=0.0361  days=60
INFO:Strategy.model.rolling_trainer:━━ Fold 3/11: Val=[2021-07-01, 2021-09-30]  Train=IS Train-Val ━━
INFO:Strategy.model.rolling_trainer:  Train: 3157132 行 (593 日) | Val: 340736 行 (64 日)
INFO:Strategy.model.trainer:特征数=134
INFO:Strategy.model.trainer:训练样本 n=2772423; label std=0.034786; 特征 NaN=0.15%
INFO:Strategy.model.trainer:使用 WPCC 自定义目标函数


...

INFO:Strategy.model.trainer:训练完成. best_iteration=84
INFO:Strategy.model.rolling_trainer:  Fold 10 Val  IC=0.0162  ICIR=0.3949  RankIC=0.0112  days=59
INFO:Strategy.model.rolling_trainer:━━ Fold 11/11: Val=[2023-07-01, 2023-09-30]  Train=IS Train-Val ━━
INFO:Strategy.model.rolling_trainer:  Train: 3157132 行 (593 日) | Val: 340736 行 (64 日)
INFO:Strategy.model.trainer:特征数=134
INFO:Strategy.model.trainer:训练样本 n=2732798; label std=0.0359459; 特征 NaN=0.16%
INFO:Strategy.model.trainer:使用 WPCC 自定义目标函数


[0]	train-rmse:0.50023	train-wpcc:-0.03961	val-rmse:0.50197	val-wpcc:-0.01553
[99]	train-rmse:0.50024	train-wpcc:-0.14046	val-rmse:0.50197	val-wpcc:-0.02916


INFO:Strategy.model.trainer:训练完成. best_iteration=19
INFO:Strategy.model.rolling_trainer:  Fold 11 Val  IC=0.0292  ICIR=0.6610  RankIC=0.0140  days=64
INFO:Strategy.model.rolling_trainer:Rolling Val CV 完成: 11 Fold 已训练
INFO:Strategy.model.rolling_trainer:滚动模型已保存: /root/autodl-tmp/Strategy/outputs/scores/rolling_xgb_OPEN930_1000.pkl (11 folds)



========== XGB CV Fold IC ==========
    fold_id status  val_ic_mean  val_ic_std  val_icir  val_rank_ic_mean  \
0         1     ok     0.042063    0.048765  0.862560          0.057517   
1         2     ok     0.052324    0.053744  0.973569          0.036146   
2         3     ok     0.038771    0.041364  0.937310          0.061444   
3         4     ok     0.026984    0.053423  0.505113          0.022174   
4         5     ok     0.038480    0.052307  0.735662          0.041406   
5         6     ok     0.033855    0.047459  0.713367          0.031399   
6         7     ok     0.030309    0.051569  0.587735          0.032081   
7         8     ok     0.038534    0.050316  0.765848          0.053016   
8         9     ok     0.031994    0.054613  0.585830          0.036775   
9        10     ok     0.016192    0.040999  0.394928          0.011176   
10       11     ok     0.029160    0.044115  0.660991          0.013958   

    n_val_days  
0           48  
1           60  
2         

PosixPath('/root/autodl-tmp/Strategy/outputs/scores/rolling_xgb_OPEN930_1000.pkl')

In [7]:
from Strategy.model.transformer_trainer import TransformerTrainer

rt_tf = RollingTrainer(
    model_class=TransformerTrainer,
    model_kwargs={
        "d_model": 64, "nhead": 4, "num_layers": 2, "d_ff": 128, "dropout": 0.15,
        "epochs": 50, "lr": 5e-4, "weight_decay": 0.01, "batch_size": 1,
        "early_stopping_patience": 8, "device": "cuda", "use_amp": False,
    }
)

rt_tf.train_all_folds(is_train)

print("\n========== Transformer CV Fold IC ==========")
print(rt_tf.fold_ic_report())
rt_tf.save_model(config.SCORE_OUTPUT_DIR / f"rolling_transformer_{LABEL_TAG}.pkl")


INFO:Strategy.model.rolling_trainer:generate_folds: IS Train=[2021-01-01, 2023-09-30], val_months=3 → 11 Fold
INFO:Strategy.model.rolling_trainer:Rolling Val CV: 11 Fold, 特征数=134, IS Train=2021-01-01~2023-09-30
INFO:Strategy.model.rolling_trainer:━━ Fold 1/11: Val=[2021-01-01, 2021-03-31]  Train=IS Train-Val ━━
INFO:Strategy.model.rolling_trainer:  Train: 3242316 行 (609 日) | Val: 255552 行 (48 日)
INFO:Strategy.model.transformer_trainer:训练设备: cuda, AMP: False
INFO:Strategy.model.transformer_trainer:CrossSectionalTransformer: d_input=134 d_model=64 nhead=4 layers=2 params=0.08M
INFO:Strategy.model.transformer_trainer:训练集: 609 日 | 验证集: 48 日 | 特征: 134
INFO:Strategy.model.transformer_trainer:  Epoch 1/50  train=-0.00137  val=-0.02230  lr=5.00e-04
INFO:Strategy.model.transformer_trainer:  Epoch 2/50  train=-0.00185  val=-0.02046  lr=4.98e-04
INFO:Strategy.model.transformer_trainer:  Epoch 3/50  train=-0.00348  val=-0.02374  lr=4.96e-04
INFO:Strategy.model.transformer_trainer:  Epoch 5/50  tra


========== Transformer CV Fold IC ==========
    fold_id status  val_ic_mean  val_ic_std  val_icir  val_rank_ic_mean  \
0         1     ok     0.025144    0.084533  0.297440          0.030985   
1         2     ok     0.012536    0.043057  0.291154          0.017966   
2         3     ok     0.030491    0.060653  0.502714          0.041673   
3         4     ok     0.006633    0.070342  0.094294         -0.006820   
4         5     ok    -0.001742    0.043570 -0.039974          0.002268   
5         6     ok     0.027639    0.066710  0.414316          0.016280   
6         7     ok     0.009480    0.069351  0.136701         -0.007990   
7         8     ok     0.013459    0.052451  0.256595          0.012108   
8         9     ok     0.017529    0.058732  0.298452          0.025399   
9        10     ok    -0.016039    0.076555 -0.209509         -0.028514   
10       11     ok    -0.001165    0.073408 -0.015864         -0.007699   

    n_val_days  
0           48  
1           60  
2 

PosixPath('/root/autodl-tmp/Strategy/outputs/scores/rolling_transformer_OPEN930_1000.pkl')

In [8]:
from Strategy.model.mlp_trainer import MLPTrainer

rt_mlp = RollingTrainer(
    model_class=MLPTrainer,
    model_kwargs={
        "hidden_dims": (256, 128, 64),
        "dropout": 0.15,
        "epochs": 50,
        "lr": 5e-4,
        "weight_decay": 0.01,
        "batch_size": 1,
        "early_stopping_patience": 8,
        "device": "cuda",
        "use_amp": False,
    },
)

rt_mlp.train_all_folds(is_train)

print("\n========== MLP CV Fold IC ==========")
print(rt_mlp.fold_ic_report())
rt_mlp.save_model(config.SCORE_OUTPUT_DIR / f"rolling_mlp_{LABEL_TAG}.pkl")


INFO:Strategy.model.rolling_trainer:generate_folds: IS Train=[2021-01-01, 2023-09-30], val_months=3 → 11 Fold
INFO:Strategy.model.rolling_trainer:Rolling Val CV: 11 Fold, 特征数=134, IS Train=2021-01-01~2023-09-30
INFO:Strategy.model.rolling_trainer:━━ Fold 1/11: Val=[2021-01-01, 2021-03-31]  Train=IS Train-Val ━━
INFO:Strategy.model.rolling_trainer:  Train: 3242316 行 (609 日) | Val: 255552 行 (48 日)
INFO:Strategy.model.mlp_trainer:MLP 训练设备: cuda, AMP: False
INFO:Strategy.model.mlp_trainer:CrossSectionalMLP: d_input=134 hidden=(256, 128, 64) params=0.08M
INFO:Strategy.model.mlp_trainer:训练集: 609 日 | 验证集: 48 日 | 特征: 134
INFO:Strategy.model.mlp_trainer:  Epoch 1/50  train=0.00146  val=-0.02213  lr=5.00e-04
INFO:Strategy.model.mlp_trainer:  Epoch 2/50  train=-0.00428  val=-0.01564  lr=4.98e-04
INFO:Strategy.model.mlp_trainer:  Epoch 3/50  train=-0.00296  val=-0.01662  lr=4.96e-04
INFO:Strategy.model.mlp_trainer:  Epoch 5/50  train=-0.00285  val=-0.01385  lr=4.88e-04
INFO:Strategy.model.mlp_trai


========== MLP CV Fold IC ==========
    fold_id status  val_ic_mean  val_ic_std  val_icir  val_rank_ic_mean  \
0         1     ok     0.023571    0.085337  0.276209          0.027539   
1         2     ok     0.013126    0.041155  0.318931          0.013154   
2         3     ok     0.011300    0.107047  0.105558          0.042784   
3         4     ok     0.003975    0.072860  0.054555         -0.002178   
4         5     ok    -0.007124    0.039672 -0.179566          0.000692   
5         6     ok     0.026525    0.062477  0.424553          0.016049   
6         7     ok     0.017366    0.079510  0.218416          0.021644   
7         8     ok     0.018557    0.072701  0.255250          0.032170   
8         9     ok    -0.001401    0.037053 -0.037800         -0.004852   
9        10     ok    -0.013948    0.082714 -0.168631          0.003723   
10       11     ok     0.000004    0.070149  0.000053          0.011626   

    n_val_days  
0           48  
1           60  
2         

PosixPath('/root/autodl-tmp/Strategy/outputs/scores/rolling_mlp_OPEN930_1000.pkl')

## 4. IS Test 核心推理与模型集成
利用最近的 4-Fold 模型对 `is_test` 进行打分，随后进行跨模型种类（XGB / Transformer / MLP）的集成。


In [12]:
from Strategy.model.scorer import mask_scores_by_price
from Strategy.data_io.saver import save_wide_table
from Strategy.model.ensemble_scorer import compute_score_correlation, select_ensemble_models, ensemble_scores

score_test_xgb = rt_xgb.predict_is_test(is_test, normalize=True)
score_test_xgb = mask_scores_by_price(score_test_xgb, label_tag=LABEL_TAG)
save_wide_table(score_test_xgb, config.SCORE_OUTPUT_DIR / f"SCORE_xgb_{LABEL_TAG}_is_test.fea")

score_test_tf = rt_tf.predict_is_test(is_test, normalize=True)
score_test_tf = mask_scores_by_price(score_test_tf, label_tag=LABEL_TAG)
save_wide_table(score_test_tf, config.SCORE_OUTPUT_DIR / f"SCORE_transformer_{LABEL_TAG}_is_test.fea")

score_test_mlp = rt_mlp.predict_is_test(is_test, normalize=True)
score_test_mlp = mask_scores_by_price(score_test_mlp, label_tag=LABEL_TAG)
save_wide_table(score_test_mlp, config.SCORE_OUTPUT_DIR / f"SCORE_mlp_{LABEL_TAG}_is_test.fea")

score_dfs = {
    "xgb": score_test_xgb,
    "transformer": score_test_tf,
    "mlp": score_test_mlp,
}

corr = compute_score_correlation(score_dfs)
print("\n模型打分截面相关性矩阵:")
print(corr)

selected = select_ensemble_models(score_dfs, max_pairwise_corr=0.9)

score_test_ens = ensemble_scores(
    score_dfs,
    selected_models=selected,
    label_tag=f"{LABEL_TAG}_is_test",
    save=True,
    output_dir=config.SCORE_OUTPUT_DIR
)


INFO:Strategy.model.rolling_trainer:IS Test Ensemble: 选取最近 4 个 Fold → Fold IDs=[11, 10, 9, 8]  Val Ends=[datetime.date(2023, 9, 30), datetime.date(2023, 6, 30), datetime.date(2023, 3, 31), datetime.date(2022, 12, 31)]
INFO:Strategy.model.rolling_trainer:  Fold 11 推理完成: mean=0.5000 std=0.0004
INFO:Strategy.model.rolling_trainer:  Fold 10 推理完成: mean=0.5001 std=0.0004
INFO:Strategy.model.rolling_trainer:  Fold 9 推理完成: mean=0.5001 std=0.0005
INFO:Strategy.model.rolling_trainer:  Fold 8 推理完成: mean=0.5001 std=0.0003
INFO:Strategy.model.rolling_trainer:IS Test Ensemble 推理完成: 222 dates × 5324 stocks
INFO:Strategy.model.scorer:price mask: before=1181928 after=1130010 removed=51918
INFO:Strategy.model.rolling_trainer:IS Test Ensemble: 选取最近 4 个 Fold → Fold IDs=[11, 10, 9, 8]  Val Ends=[datetime.date(2023, 9, 30), datetime.date(2023, 6, 30), datetime.date(2023, 3, 31), datetime.date(2022, 12, 31)]
INFO:Strategy.model.rolling_trainer:  Fold 11 推理完成: mean=-2.3182 std=4.0186
INFO:Strategy.model.rolli


模型打分截面相关性矩阵:
                  mlp  transformer       xgb
mlp          1.000000     0.866525  0.100402
transformer  0.866525     1.000000  0.105704
xgb          0.100402     0.105704  1.000000


INFO:Strategy.model.ensemble_scorer:模型打分相关性矩阵 (222 日均):
               mlp  transformer    xgb
mlp          1.000        0.867  0.100
transformer  0.867        1.000  0.106
xgb          0.100        0.106  1.000
INFO:Strategy.model.ensemble_scorer:跨模型集成入选: ['xgb', 'transformer', 'mlp'] (3 / 3)
INFO:Strategy.model.ensemble_scorer:跨模型集成: 3 个模型 → ['xgb', 'transformer', 'mlp']
/root/autodl-tmp/Strategy/model/ensemble_scorer.py:256: RuntimeWarning: Mean of empty slice
  avg_scores = np.nanmean(stacked, axis=0)                       # (dates, stocks)
INFO:Strategy.model.ensemble_scorer:跨模型集成打分: 222 dates × 5324 stocks
INFO:Strategy.model.ensemble_scorer:跨模型集成打分已保存: /root/autodl-tmp/Strategy/outputs/scores/SCORE_ensemble_OPEN930_1000_is_test.fea


## 5. 快速业绩回测 (仅观测 IS_Test)
在没有发生前视偏差的数据区间上观测策略真实的盈利能力。

In [13]:
import sys
import logging
from pathlib import Path

ROOT = Path("/root/autodl-tmp/Strategy")
if str(ROOT.parent) not in sys.path:
    sys.path.insert(0, str(ROOT.parent))
logging.basicConfig(level=logging.INFO, force=True)

from Strategy import config
from Strategy.label.label_generator import load_label
from Strategy.model.scorer import load_is_test_scores_from_disk
from Strategy.backtest.quick_backtest import run_quick_backtest

LABEL_TAG = "OPEN930_1000"

label_df = load_label(LABEL_TAG)
models_to_test = load_is_test_scores_from_disk(LABEL_TAG)

for name, score_df in models_to_test.items():
    print(f"\n{'='*50}\n回测评估: {name} (IS_Test)\n{'='*50}")
    run_quick_backtest(
        score_df=score_df,
        label_df=label_df,
        n_groups=20,
        output_dir=config.BT_RESULT_DIR / "is_test" / name,
        start_date=config.IS_TEST_START,
        top_ks=(20, 50, 100),
        tail_ks=(20, 50, 100),
        run_ic=True,
        title=f"{name.upper()} | {LABEL_TAG} | IS_Test",
    )


INFO:Strategy.model.scorer:price mask: before=1130010 after=1130010 removed=0
INFO:Strategy.model.scorer:price mask: before=1130010 after=1130010 removed=0
INFO:Strategy.model.scorer:price mask: before=1130010 after=1130010 removed=0
INFO:Strategy.model.scorer:price mask: before=1130010 after=1130010 removed=0



回测评估: xgb (IS_Test)


INFO:Strategy.backtest.quick_backtest:分层回测完成: 222 交易日, 20 组, topK=(20, 50, 100), tailK=(20, 50, 100)
INFO:Strategy.backtest.quick_backtest:stock pool: score=5090 tradeable=3009 no_twap=0 limit_up=32 new=6 delist=0 st=564 prefix=1477
INFO:Strategy.backtest.quick_backtest:  group1: Ann=-40.69% MDD=-50.81% SR=-0.87 WR=47.30%
INFO:Strategy.backtest.quick_backtest:  group2: Ann=-10.34% MDD=-47.01% SR=-0.23 WR=50.00%
INFO:Strategy.backtest.quick_backtest:  group3: Ann=-9.42% MDD=-44.07% SR=-0.23 WR=47.75%
INFO:Strategy.backtest.quick_backtest:  group4: Ann=-13.67% MDD=-45.44% SR=-0.34 WR=49.55%
INFO:Strategy.backtest.quick_backtest:  group5: Ann=-11.43% MDD=-43.57% SR=-0.30 WR=50.90%
INFO:Strategy.backtest.quick_backtest:  group6: Ann=-15.92% MDD=-42.76% SR=-0.42 WR=50.00%
INFO:Strategy.backtest.quick_backtest:  group7: Ann=-22.67% MDD=-42.13% SR=-0.64 WR=49.10%
INFO:Strategy.backtest.quick_backtest:  group8: Ann=-21.71% MDD=-41.27% SR=-0.63 WR=46.85%
INFO:Strategy.backtest.quick_backtest:  


回测评估: transformer (IS_Test)


INFO:Strategy.backtest.quick_backtest:分层回测完成: 222 交易日, 20 组, topK=(20, 50, 100), tailK=(20, 50, 100)
INFO:Strategy.backtest.quick_backtest:stock pool: score=5090 tradeable=3009 no_twap=0 limit_up=32 new=6 delist=0 st=564 prefix=1477
INFO:Strategy.backtest.quick_backtest:  group1: Ann=-19.09% MDD=-41.96% SR=-0.53 WR=44.59%
INFO:Strategy.backtest.quick_backtest:  group2: Ann=-17.98% MDD=-41.90% SR=-0.52 WR=43.24%
INFO:Strategy.backtest.quick_backtest:  group3: Ann=-21.66% MDD=-42.13% SR=-0.63 WR=45.95%
INFO:Strategy.backtest.quick_backtest:  group4: Ann=-15.90% MDD=-41.34% SR=-0.47 WR=47.75%
INFO:Strategy.backtest.quick_backtest:  group5: Ann=-24.98% MDD=-43.37% SR=-0.75 WR=44.59%
INFO:Strategy.backtest.quick_backtest:  group6: Ann=-29.17% MDD=-43.35% SR=-0.88 WR=47.30%
INFO:Strategy.backtest.quick_backtest:  group7: Ann=-27.85% MDD=-40.90% SR=-0.87 WR=46.40%
INFO:Strategy.backtest.quick_backtest:  group8: Ann=-29.36% MDD=-39.21% SR=-0.93 WR=47.30%
INFO:Strategy.backtest.quick_backtest: 


回测评估: mlp (IS_Test)


INFO:Strategy.backtest.quick_backtest:分层回测完成: 222 交易日, 20 组, topK=(20, 50, 100), tailK=(20, 50, 100)
INFO:Strategy.backtest.quick_backtest:stock pool: score=5090 tradeable=3009 no_twap=0 limit_up=32 new=6 delist=0 st=564 prefix=1477
INFO:Strategy.backtest.quick_backtest:  group1: Ann=-19.13% MDD=-27.93% SR=-0.68 WR=46.40%
INFO:Strategy.backtest.quick_backtest:  group2: Ann=-13.93% MDD=-31.85% SR=-0.45 WR=46.40%
INFO:Strategy.backtest.quick_backtest:  group3: Ann=-19.96% MDD=-37.13% SR=-0.62 WR=47.30%
INFO:Strategy.backtest.quick_backtest:  group4: Ann=-18.41% MDD=-41.08% SR=-0.54 WR=46.40%
INFO:Strategy.backtest.quick_backtest:  group5: Ann=-14.94% MDD=-44.17% SR=-0.43 WR=45.05%
INFO:Strategy.backtest.quick_backtest:  group6: Ann=-32.10% MDD=-44.46% SR=-0.93 WR=42.34%
INFO:Strategy.backtest.quick_backtest:  group7: Ann=-34.27% MDD=-44.19% SR=-1.01 WR=44.59%
INFO:Strategy.backtest.quick_backtest:  group8: Ann=-30.83% MDD=-43.20% SR=-0.90 WR=44.59%
INFO:Strategy.backtest.quick_backtest: 


回测评估: ensemble (IS_Test)


INFO:Strategy.backtest.quick_backtest:分层回测完成: 222 交易日, 20 组, topK=(20, 50, 100), tailK=(20, 50, 100)
INFO:Strategy.backtest.quick_backtest:stock pool: score=5090 tradeable=3009 no_twap=0 limit_up=32 new=6 delist=0 st=564 prefix=1477
INFO:Strategy.backtest.quick_backtest:  group1: Ann=-16.46% MDD=-47.50% SR=-0.37 WR=48.20%
INFO:Strategy.backtest.quick_backtest:  group2: Ann=-22.63% MDD=-45.19% SR=-0.58 WR=47.75%
INFO:Strategy.backtest.quick_backtest:  group3: Ann=-21.85% MDD=-43.11% SR=-0.61 WR=48.20%
INFO:Strategy.backtest.quick_backtest:  group4: Ann=-24.85% MDD=-43.02% SR=-0.75 WR=47.30%
INFO:Strategy.backtest.quick_backtest:  group5: Ann=-24.91% MDD=-40.19% SR=-0.81 WR=46.40%
INFO:Strategy.backtest.quick_backtest:  group6: Ann=-26.73% MDD=-39.44% SR=-0.87 WR=45.95%
INFO:Strategy.backtest.quick_backtest:  group7: Ann=-35.50% MDD=-38.33% SR=-1.16 WR=44.14%
INFO:Strategy.backtest.quick_backtest:  group8: Ann=-27.81% MDD=-35.90% SR=-0.89 WR=47.30%
INFO:Strategy.backtest.quick_backtest: 

## 6. OOS 盲区验证 (严格门控)
⚠️ **重要提示**：此部分代码仅当上方 `IS_Test` 回测结果各项指标完全达到预期目标，决定实盘部署时，才允许运行！一旦针对 OOS 数据调整超参，OOS 就被“污染”了。

In [11]:
# score_oos_xgb = rt_xgb.predict_is_test(oos, normalize=True)
# score_oos_tf = rt_tf.predict_is_test(oos, normalize=True)
# score_oos_mlp = rt_mlp.predict_is_test(oos, normalize=True)
# score_oos_xgb = mask_scores_by_price(score_oos_xgb, label_tag=LABEL_TAG)
# score_oos_tf = mask_scores_by_price(score_oos_tf, label_tag=LABEL_TAG)
# score_oos_mlp = mask_scores_by_price(score_oos_mlp, label_tag=LABEL_TAG)
#
# score_oos_ens = ensemble_scores(
#     {"xgb": score_oos_xgb, "transformer": score_oos_tf, "mlp": score_oos_mlp},
#     selected_models=selected,
#     label_tag=f"{LABEL_TAG}_oos",
#     save=True,
#     output_dir=config.SCORE_OUTPUT_DIR
# )
#
# run_quick_backtest(
#     score_df=score_oos_ens,
#     label_df=label_df,
#     n_groups=20,
#     output_dir=config.BT_RESULT_DIR / "oos" / "ensemble",
#     start_date=config.OOS_START,
#     top_ks=(20, 50, 100),
#     tail_ks=(20, 50, 100),
#     run_ic=True,
#     title=f"ENSEMBLE | {LABEL_TAG} | OOS"
# )
